# Phase 0: Mathematical & Computational Foundations  
## Notebook 00_05 — From Python Prototype to C++ Finance Kernel

### Research question

How can a quantitative finance prototype written in Python be converted into a compiled C++ kernel while preserving correctness, reproducibility, and usability from a Python research workflow?

This notebook builds a small but realistic finance kernel:

1. generate synthetic multi-asset returns and target weights;
2. implement a transaction-cost-aware portfolio backtest in pure Python;
3. implement the same logic in vectorised NumPy;
4. write a compiled C++ kernel;
5. call the C++ kernel from Python using `ctypes`;
6. benchmark the implementations;
7. validate numerical equivalence;
8. discuss when C++ is worth the engineering cost.

The goal is not to replace Python. The goal is to use Python for research orchestration and C++ for performance-critical inner loops.

## 1. Why this matters in quantitative finance

A common research workflow is:

1. prototype quickly in Python;
2. validate the mathematics and data logic;
3. profile the code;
4. move only the bottleneck into a compiled kernel;
5. keep the outer research workflow in Python.

This hybrid approach is useful because finance code often contains both:

- **high-level experimentation**, where Python is excellent;
- **tight numerical loops**, where compiled code can be faster and more predictable.

Examples of finance kernels that may benefit from compiled implementation include:

- Monte Carlo path simulation;
- option pricing inner loops;
- order book event replay;
- transaction-cost-aware backtesting;
- path-dependent payoff evaluation;
- risk aggregation across many scenarios;
- high-frequency feature computation.

This notebook uses a stateful portfolio backtest kernel because it is simple enough to audit but realistic enough to show why implementation details matter.

## 2. Kernel specification

We consider a portfolio with $N$ assets over $T$ time steps.

Inputs:

- $R_{t,i}$: return of asset $i$ at time $t$;
- $w_{t,i}$: target portfolio weight of asset $i$ at time $t$;
- $c$: proportional transaction cost rate per unit of turnover.

At each time step:

1. rebalance from previous weights $w_{t-1}$ to current target weights $w_t$;
2. pay transaction costs proportional to turnover;
3. earn the weighted portfolio return.

Turnover is:

$$
\text{turnover}_t = \sum_{i=1}^{N} |w_{t,i} - w_{t-1,i}|
$$

Gross portfolio return is:

$$
r^{\text{gross}}_t = \sum_{i=1}^{N} w_{t,i}R_{t,i}
$$

Net portfolio return is:

$$
r^{\text{net}}_t = r^{\text{gross}}_t - c \cdot \text{turnover}_t
$$

The equity curve evolves as:

$$
E_t = E_{t-1}(1 + r^{\text{net}}_t)
$$

This is intentionally simple, but it contains the key features of many larger backtesting systems:

- matrix-like inputs;
- repeated dot products;
- stateful turnover calculation;
- cumulative equity update;
- numerical validation requirements.

In [ ]:
import ctypes
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

## 3. Synthetic data generation

To keep the notebook reproducible, we generate synthetic returns and slowly changing target weights.

The return model has a simple factor structure:

$$
r_t = Bf_t + \epsilon_t
$$

The target weights are generated from smooth random scores and transformed into long-only weights that sum to one.

This gives us a controlled benchmark dataset without relying on external market data.

In [ ]:
def generate_synthetic_returns(
    num_steps: int,
    num_assets: int,
    num_factors: int,
    seed: int = 42
) -> np.ndarray:
    """
    Generate synthetic multi-asset returns using a simple factor model.
    """
    local_rng = np.random.default_rng(seed)

    loadings = local_rng.normal(loc=0.0, scale=0.7, size=(num_assets, num_factors))

    factor_vols = np.linspace(0.012, 0.006, num_factors)
    factor_returns = local_rng.normal(
        loc=0.0,
        scale=factor_vols,
        size=(num_steps, num_factors)
    )

    idio_vols = local_rng.uniform(0.004, 0.014, size=num_assets)
    noise = local_rng.normal(
        loc=0.0,
        scale=idio_vols,
        size=(num_steps, num_assets)
    )

    returns = factor_returns @ loadings.T + noise

    return np.ascontiguousarray(returns, dtype=np.float64)


def softmax_stable(x: np.ndarray, axis: int = 1) -> np.ndarray:
    """
    Numerically stable softmax.
    """
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / exp_shifted.sum(axis=axis, keepdims=True)


def generate_smooth_target_weights(
    num_steps: int,
    num_assets: int,
    seed: int = 123
) -> np.ndarray:
    """
    Generate slowly changing long-only target weights.

    The weights are not intended to be alpha signals. They simply create
    a realistic stateful input for the backtest kernel.
    """
    local_rng = np.random.default_rng(seed)

    raw_shocks = local_rng.normal(loc=0.0, scale=0.15, size=(num_steps, num_assets))
    scores = np.zeros_like(raw_shocks)

    persistence = 0.97

    for t in range(1, num_steps):
        scores[t] = persistence * scores[t - 1] + raw_shocks[t]

    weights = softmax_stable(scores, axis=1)

    return np.ascontiguousarray(weights, dtype=np.float64)

In [ ]:
num_steps = 7_500
num_assets = 120
num_factors = 6
cost_rate = 0.0005  # 5 basis points per unit of turnover

returns = generate_synthetic_returns(
    num_steps=num_steps,
    num_assets=num_assets,
    num_factors=num_factors
)

weights = generate_smooth_target_weights(
    num_steps=num_steps,
    num_assets=num_assets
)

returns.shape, weights.shape, weights.sum(axis=1)[:5]

## 4. Pure Python implementation

The pure Python implementation is written in a deliberately explicit way.

This makes the logic easy to audit, but it is not the fastest implementation because it loops over both time and assets in Python.

In [ ]:
def backtest_python_loop(
    returns: np.ndarray,
    weights: np.ndarray,
    cost_rate: float
) -> dict:
    """
    Pure Python loop implementation of the portfolio backtest.

    This version prioritises readability over speed.
    """
    num_steps, num_assets = returns.shape

    net_returns = np.zeros(num_steps, dtype=np.float64)
    turnover = np.zeros(num_steps, dtype=np.float64)
    equity = np.zeros(num_steps, dtype=np.float64)

    previous_weights = [0.0] * num_assets
    current_equity = 1.0
    running_max = 1.0
    max_drawdown = 0.0

    for t in range(num_steps):
        current_turnover = 0.0
        gross_return = 0.0

        for i in range(num_assets):
            current_turnover += abs(float(weights[t, i]) - previous_weights[i])
            gross_return += float(weights[t, i]) * float(returns[t, i])

        current_net_return = gross_return - cost_rate * current_turnover
        current_equity *= (1.0 + current_net_return)

        running_max = max(running_max, current_equity)
        drawdown = current_equity / running_max - 1.0
        max_drawdown = min(max_drawdown, drawdown)

        net_returns[t] = current_net_return
        turnover[t] = current_turnover
        equity[t] = current_equity

        previous_weights = [float(weights[t, i]) for i in range(num_assets)]

    return {
        "net_returns": net_returns,
        "turnover": turnover,
        "equity": equity,
        "max_drawdown": max_drawdown,
        "average_turnover": float(turnover.mean())
    }

## 5. Vectorised NumPy implementation

The NumPy implementation uses array operations to move most work into compiled NumPy internals.

This is often the correct first optimisation step before writing custom C++.

The key operations are:

$$
r^{\text{gross}}_t = \sum_i w_{t,i}R_{t,i}
$$

and:

$$
\text{turnover}_t = \sum_i |w_{t,i} - w_{t-1,i}|
$$

In [ ]:
def backtest_numpy_vectorized(
    returns: np.ndarray,
    weights: np.ndarray,
    cost_rate: float
) -> dict:
    """
    Vectorised NumPy implementation of the portfolio backtest.
    """
    previous_weights = np.vstack([
        np.zeros((1, weights.shape[1])),
        weights[:-1]
    ])

    turnover = np.sum(np.abs(weights - previous_weights), axis=1)
    gross_returns = np.sum(weights * returns, axis=1)
    net_returns = gross_returns - cost_rate * turnover

    equity = np.cumprod(1.0 + net_returns)
    running_max = np.maximum.accumulate(np.insert(equity, 0, 1.0))[1:]
    drawdown = equity / running_max - 1.0

    return {
        "net_returns": net_returns,
        "turnover": turnover,
        "equity": equity,
        "max_drawdown": float(drawdown.min()),
        "average_turnover": float(turnover.mean())
    }

In [ ]:
python_result = backtest_python_loop(returns, weights, cost_rate)
numpy_result = backtest_numpy_vectorized(returns, weights, cost_rate)

np.testing.assert_allclose(
    python_result["net_returns"],
    numpy_result["net_returns"],
    rtol=1e-12,
    atol=1e-12
)

np.testing.assert_allclose(
    python_result["equity"],
    numpy_result["equity"],
    rtol=1e-12,
    atol=1e-12
)

print("Pure Python and NumPy implementations match.")

## 6. Writing a C++ kernel

For the C++ version, we write a small shared library exposing a C-compatible function.

The function receives raw pointers to contiguous `float64` arrays:

- `returns`: flattened $T \times N$ return matrix;
- `weights`: flattened $T \times N$ target weight matrix;
- `net_returns_out`: output array;
- `turnover_out`: output array;
- `equity_out`: output array;
- `summary_out`: output summary statistics.

This notebook uses `ctypes` to avoid requiring a full Python package build.

In a production-quality Python package, a cleaner approach would usually be:

- `pybind11` for C++ bindings;
- `scikit-build-core` or CMake for packaging;
- unit tests and continuous integration;
- explicit memory-layout and dtype checks.

In [ ]:
cpp_source = r"""
#include <cmath>
#include <cstddef>
#include <algorithm>

extern "C" void portfolio_backtest_kernel(
    const double* returns,
    const double* weights,
    int num_steps,
    int num_assets,
    double cost_rate,
    double* net_returns_out,
    double* turnover_out,
    double* equity_out,
    double* summary_out
) {
    double current_equity = 1.0;
    double running_max = 1.0;
    double max_drawdown = 0.0;
    double turnover_sum = 0.0;

    for (int t = 0; t < num_steps; ++t) {
        double gross_return = 0.0;
        double turnover = 0.0;

        for (int i = 0; i < num_assets; ++i) {
            const int idx = t * num_assets + i;
            const double current_weight = weights[idx];
            const double asset_return = returns[idx];

            double previous_weight = 0.0;

            if (t > 0) {
                const int prev_idx = (t - 1) * num_assets + i;
                previous_weight = weights[prev_idx];
            }

            turnover += std::abs(current_weight - previous_weight);
            gross_return += current_weight * asset_return;
        }

        const double net_return = gross_return - cost_rate * turnover;
        current_equity *= (1.0 + net_return);

        running_max = std::max(running_max, current_equity);
        const double drawdown = current_equity / running_max - 1.0;
        max_drawdown = std::min(max_drawdown, drawdown);

        net_returns_out[t] = net_return;
        turnover_out[t] = turnover;
        equity_out[t] = current_equity;

        turnover_sum += turnover;
    }

    summary_out[0] = current_equity;
    summary_out[1] = max_drawdown;
    summary_out[2] = turnover_sum / static_cast<double>(num_steps);
}
"""

build_dir = Path("build_cpp_kernel")
build_dir.mkdir(exist_ok=True)

cpp_path = build_dir / "portfolio_backtest_kernel.cpp"
cpp_path.write_text(cpp_source)

cpp_path

## 7. Compiling the C++ shared library

The next cell attempts to compile the C++ source file.

This requires a local C++ compiler such as `g++`, `clang++`, or Microsoft Visual C++.

If no compiler is available, the notebook will skip the C++ benchmark and still run the Python and NumPy sections.

In [ ]:
def find_cpp_compiler() -> str | None:
    """
    Find a usable C++ compiler on the local system.
    """
    candidates = ["c++", "g++", "clang++"]

    for compiler in candidates:
        if shutil.which(compiler) is not None:
            return compiler

    return None


def compile_shared_library(cpp_path: Path, build_dir: Path) -> Path | None:
    """
    Compile the C++ kernel into a shared library.

    Returns the shared library path if successful, otherwise None.
    """
    compiler = find_cpp_compiler()

    if compiler is None:
        print("No C++ compiler found. Skipping C++ compilation.")
        return None

    system = platform.system()

    if system == "Darwin":
        lib_path = build_dir / "libportfolio_kernel.dylib"
        command = [
            compiler,
            "-O3",
            "-std=c++17",
            "-shared",
            "-fPIC",
            str(cpp_path),
            "-o",
            str(lib_path)
        ]
    elif system == "Linux":
        lib_path = build_dir / "libportfolio_kernel.so"
        command = [
            compiler,
            "-O3",
            "-std=c++17",
            "-shared",
            "-fPIC",
            str(cpp_path),
            "-o",
            str(lib_path)
        ]
    elif system == "Windows":
        print("Windows shared-library compilation is compiler-specific. Skipping automatic compilation.")
        return None
    else:
        print(f"Unsupported platform for automatic compilation: {system}")
        return None

    print("Compiler:", compiler)
    print("Command:", " ".join(command))

    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as exc:
        print("Compilation failed.")
        print(exc.stdout)
        print(exc.stderr)
        return None

    return lib_path


lib_path = compile_shared_library(cpp_path, build_dir)
lib_path

## 8. Calling the C++ kernel from Python

If compilation succeeded, we load the shared library with `ctypes`.

Before crossing the Python/C++ boundary, we enforce:

1. `float64` dtype;
2. C-contiguous memory layout;
3. matching shapes;
4. preallocated output arrays.

This is not glamour work, but it is where many bugs happen in real numerical systems.

In [ ]:
def backtest_cpp_ctypes(
    returns: np.ndarray,
    weights: np.ndarray,
    cost_rate: float,
    lib_path: Path | None
) -> dict | None:
    """
    Call the compiled C++ backtest kernel using ctypes.
    """
    if lib_path is None:
        print("C++ shared library not available. Skipping C++ call.")
        return None

    returns_c = np.ascontiguousarray(returns, dtype=np.float64)
    weights_c = np.ascontiguousarray(weights, dtype=np.float64)

    if returns_c.shape != weights_c.shape:
        raise ValueError("returns and weights must have the same shape.")

    num_steps, num_assets = returns_c.shape

    net_returns_out = np.empty(num_steps, dtype=np.float64)
    turnover_out = np.empty(num_steps, dtype=np.float64)
    equity_out = np.empty(num_steps, dtype=np.float64)
    summary_out = np.empty(3, dtype=np.float64)

    lib = ctypes.CDLL(str(lib_path.resolve()))
    kernel = lib.portfolio_backtest_kernel

    kernel.argtypes = [
        ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_double),
        ctypes.c_int,
        ctypes.c_int,
        ctypes.c_double,
        ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_double),
        ctypes.POINTER(ctypes.c_double),
    ]

    kernel.restype = None

    kernel(
        returns_c.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        weights_c.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        ctypes.c_int(num_steps),
        ctypes.c_int(num_assets),
        ctypes.c_double(cost_rate),
        net_returns_out.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        turnover_out.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        equity_out.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        summary_out.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
    )

    return {
        "net_returns": net_returns_out,
        "turnover": turnover_out,
        "equity": equity_out,
        "final_equity": float(summary_out[0]),
        "max_drawdown": float(summary_out[1]),
        "average_turnover": float(summary_out[2])
    }


cpp_result = backtest_cpp_ctypes(returns, weights, cost_rate, lib_path)
cpp_result is not None

In [ ]:
if cpp_result is not None:
    np.testing.assert_allclose(
        cpp_result["net_returns"],
        numpy_result["net_returns"],
        rtol=1e-12,
        atol=1e-12
    )

    np.testing.assert_allclose(
        cpp_result["turnover"],
        numpy_result["turnover"],
        rtol=1e-12,
        atol=1e-12
    )

    np.testing.assert_allclose(
        cpp_result["equity"],
        numpy_result["equity"],
        rtol=1e-12,
        atol=1e-12
    )

    print("C++ kernel matches NumPy implementation.")
else:
    print("C++ validation skipped because compilation was unavailable.")

## 9. Benchmarking

Benchmarks should be interpreted carefully.

The result depends on:

- hardware;
- compiler;
- BLAS implementation;
- array size;
- memory layout;
- whether the workload is vectorisable;
- overhead of crossing the Python/C++ boundary.

The correct question is not always:

> Is C++ faster?

The better question is:

> Is this specific bottleneck worth moving into compiled code?

In [ ]:
def time_function(func, repeats: int = 5) -> dict:
    """
    Time a function over several repeats and return summary statistics.
    """
    timings = []

    for _ in range(repeats):
        start = time.perf_counter()
        func()
        end = time.perf_counter()
        timings.append(end - start)

    timings = np.array(timings)

    return {
        "min_seconds": float(timings.min()),
        "median_seconds": float(np.median(timings)),
        "max_seconds": float(timings.max())
    }


benchmark_rows = []

benchmark_rows.append({
    "implementation": "pure_python_loop",
    **time_function(lambda: backtest_python_loop(returns, weights, cost_rate), repeats=3)
})

benchmark_rows.append({
    "implementation": "numpy_vectorized",
    **time_function(lambda: backtest_numpy_vectorized(returns, weights, cost_rate), repeats=10)
})

if lib_path is not None:
    benchmark_rows.append({
        "implementation": "cpp_ctypes_kernel",
        **time_function(lambda: backtest_cpp_ctypes(returns, weights, cost_rate, lib_path), repeats=10)
    })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df

In [ ]:
if len(benchmark_df) > 0:
    baseline = benchmark_df.loc[
        benchmark_df["implementation"] == "pure_python_loop",
        "median_seconds"
    ].iloc[0]

    benchmark_df["speedup_vs_python_loop"] = baseline / benchmark_df["median_seconds"]

benchmark_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(benchmark_df["implementation"], benchmark_df["median_seconds"])
plt.yscale("log")
plt.title("Median Runtime by Implementation")
plt.xlabel("Implementation")
plt.ylabel("Median seconds, log scale")
plt.xticks(rotation=30, ha="right")
plt.show()

## 10. Optional comparison: Numba JIT

Another route is to keep code in Python syntax but compile it with Numba.

This can be attractive when:

- the code is mostly numerical loops;
- the function uses NumPy arrays and scalar operations;
- fast iteration matters;
- building a C++ extension would be too heavy.

This cell is optional. It runs only if Numba is installed.

In [ ]:
try:
    from numba import njit
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False

NUMBA_AVAILABLE

In [ ]:
if NUMBA_AVAILABLE:
    @njit
    def backtest_numba_kernel(returns, weights, cost_rate):
        num_steps, num_assets = returns.shape

        net_returns = np.zeros(num_steps)
        turnover = np.zeros(num_steps)
        equity = np.zeros(num_steps)

        previous_weights = np.zeros(num_assets)
        current_equity = 1.0
        running_max = 1.0
        max_drawdown = 0.0

        for t in range(num_steps):
            current_turnover = 0.0
            gross_return = 0.0

            for i in range(num_assets):
                current_turnover += abs(weights[t, i] - previous_weights[i])
                gross_return += weights[t, i] * returns[t, i]

            current_net_return = gross_return - cost_rate * current_turnover
            current_equity *= (1.0 + current_net_return)

            if current_equity > running_max:
                running_max = current_equity

            drawdown = current_equity / running_max - 1.0

            if drawdown < max_drawdown:
                max_drawdown = drawdown

            net_returns[t] = current_net_return
            turnover[t] = current_turnover
            equity[t] = current_equity

            for i in range(num_assets):
                previous_weights[i] = weights[t, i]

        average_turnover = turnover.mean()

        return net_returns, turnover, equity, max_drawdown, average_turnover


    # First call triggers compilation.
    numba_net, numba_turnover, numba_equity, numba_mdd, numba_avg_turnover = backtest_numba_kernel(
        returns,
        weights,
        cost_rate
    )

    np.testing.assert_allclose(numba_net, numpy_result["net_returns"], rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(numba_equity, numpy_result["equity"], rtol=1e-12, atol=1e-12)

    print("Numba implementation matches NumPy implementation.")
else:
    print("Numba is not installed. Optional Numba section skipped.")

In [ ]:
if NUMBA_AVAILABLE:
    numba_timing = time_function(
        lambda: backtest_numba_kernel(returns, weights, cost_rate),
        repeats=10
    )

    numba_row = pd.DataFrame([{
        "implementation": "numba_njit",
        **numba_timing
    }])

    benchmark_with_numba_df = pd.concat([benchmark_df, numba_row], ignore_index=True)

    baseline = benchmark_with_numba_df.loc[
        benchmark_with_numba_df["implementation"] == "pure_python_loop",
        "median_seconds"
    ].iloc[0]

    benchmark_with_numba_df["speedup_vs_python_loop"] = (
        baseline / benchmark_with_numba_df["median_seconds"]
    )

    display(benchmark_with_numba_df)
else:
    benchmark_with_numba_df = benchmark_df
    display(benchmark_with_numba_df)

## 11. Correctness before speed

A fast wrong kernel is worse than a slow correct prototype.

Before trusting a compiled kernel, use:

1. **unit tests**  
   Compare against a clear Python reference implementation.

2. **shape checks**  
   Ensure input arrays have expected dimensions.

3. **dtype checks**  
   Avoid accidentally passing `float32` where `float64` is expected.

4. **memory layout checks**  
   C++ pointer arithmetic assumes a specific layout, usually C-contiguous row-major arrays.

5. **edge cases**  
   Test zero returns, zero weights, one asset, one time step, and extreme transaction costs.

6. **numerical tolerances**  
   Floating-point arithmetic may differ slightly across implementations.

7. **reproducible benchmarks**  
   Record hardware, compiler, compiler flags, array sizes, and random seeds.

In [ ]:
def run_small_kernel_tests() -> None:
    """
    Simple correctness tests for the Python and NumPy implementations.
    """
    small_returns = np.array([
        [0.01, 0.02],
        [-0.01, 0.03],
        [0.00, -0.02]
    ], dtype=np.float64)

    small_weights = np.array([
        [0.5, 0.5],
        [0.6, 0.4],
        [0.6, 0.4]
    ], dtype=np.float64)

    small_cost = 0.001

    loop_result = backtest_python_loop(small_returns, small_weights, small_cost)
    vector_result = backtest_numpy_vectorized(small_returns, small_weights, small_cost)

    np.testing.assert_allclose(loop_result["net_returns"], vector_result["net_returns"])
    np.testing.assert_allclose(loop_result["turnover"], vector_result["turnover"])
    np.testing.assert_allclose(loop_result["equity"], vector_result["equity"])

    # First-period turnover is from zero weights to the initial target weights.
    expected_first_turnover = np.abs(small_weights[0]).sum()
    assert np.isclose(vector_result["turnover"][0], expected_first_turnover)

    print("Small kernel tests passed.")


run_small_kernel_tests()

## 12. When should a quant researcher use C++?

C++ is not automatically better.

A sensible rule is:

> Start in Python, profile the bottleneck, then move only the bottleneck.

C++ is often worth considering when:

1. the workload is a tight loop over millions of events or paths;
2. the algorithm is hard to vectorise cleanly;
3. latency or deterministic runtime matters;
4. memory allocation needs careful control;
5. the same kernel will be reused many times;
6. the code is moving from research to production.

C++ may not be worth it when:

1. NumPy already calls optimised compiled libraries;
2. the bottleneck is data loading or network latency;
3. the code changes every day;
4. the performance gain is small relative to maintenance cost;
5. the team cannot properly test and maintain the compiled layer.

## 13. A production-style architecture

A realistic research-to-production path could look like this:

```text
notebooks/
    00_05_python_to_cpp_finance_kernel.ipynb

src/
    alpha_factory/
        backtest/
            python_reference.py
            validation.py

cpp/
    portfolio_backtest_kernel.cpp
    include/
        portfolio_backtest_kernel.hpp

bindings/
    pybind11_module.cpp

tests/
    test_backtest_kernel.py
    test_cpp_matches_python.py

pyproject.toml
CMakeLists.txt
```

The separation matters:

- notebooks explain and experiment;
- Python reference code provides clarity;
- C++ kernels provide speed;
- bindings expose C++ to Python;
- tests prevent silent numerical drift;
- build files make the project reproducible.

## 14. Limitations

This notebook deliberately keeps the C++ example small.

### 14.1 The kernel is simple

The backtest model ignores:

- borrow costs;
- leverage constraints;
- margin;
- shorting;
- bid-ask spread decomposition;
- nonlinear market impact;
- partial fills;
- asynchronous ticks;
- order queue priority;
- corporate actions;
- funding constraints.

These appear in later backtesting and execution notebooks.

### 14.2 `ctypes` is not ideal for large packages

`ctypes` is useful for a self-contained notebook demo, but it is not the cleanest interface for a larger C++/Python project.

For a real package, `pybind11` usually gives a more natural C++ binding style.

### 14.3 Benchmarking is environment-dependent

The speed comparison depends on:

- CPU;
- compiler;
- optimisation flags;
- NumPy build;
- memory bandwidth;
- array shapes;
- whether Numba is installed;
- whether the compiler auto-vectorises the loop.

The benchmark is therefore a workflow demonstration, not a universal performance claim.

### 14.4 Memory transfer can dominate

If data must be copied repeatedly across the Python/C++ boundary, the speed advantage of C++ can disappear.

This is why the notebook carefully uses contiguous arrays and preallocated outputs.

### 14.5 Numerical equivalence is not automatic

Different implementations can produce slightly different floating-point results because of operation ordering, compiler optimisations, and hardware instructions.

Testing must use appropriate tolerances.

## 15. Research frontier and current directions

The Python/C++ boundary is still a major design question in quantitative finance and scientific computing.

### 15.1 Python orchestration with compiled kernels

A common industry pattern is to use Python for:

- research notebooks;
- data exploration;
- model orchestration;
- plotting;
- experiment tracking.

Performance-critical kernels are then written in:

- C++;
- Rust;
- CUDA;
- JAX/XLA;
- Numba;
- Cython;
- vendor-optimised numerical libraries.

A future notebook could compare the same Monte Carlo pricer in NumPy, Numba, C++, and JAX.

### 15.2 pybind11 and modern C++ bindings

`pybind11` allows C++ functions and classes to be exposed naturally to Python.

This is useful for a larger quant codebase where C++ classes represent:

- order books;
- pricing engines;
- risk engines;
- execution simulators;
- path generators.

A future notebook could rewrite this `ctypes` example as a proper `pybind11` extension.

### 15.3 Reproducible builds with CMake and scikit-build-core

For a public GitHub repository, reproducibility matters.

A future project could package the C++ kernel using:

- `pyproject.toml`;
- CMake;
- `scikit-build-core`;
- continuous integration;
- unit tests across operating systems.

This would make the repo look more like a genuine research software project rather than a collection of notebooks.

### 15.4 Hardware acceleration and SIMD

Some finance kernels are memory-bound, while others are compute-bound.

Modern C++ can exploit:

- SIMD vectorisation;
- cache-aware memory layout;
- OpenMP threading;
- GPU acceleration;
- batched linear algebra libraries.

A future notebook could benchmark row-major versus column-major layout for a Monte Carlo path engine.

### 15.5 Differentiable and accelerator-native finance

Modern computational finance increasingly uses frameworks where the same model can be:

- simulated;
- differentiated;
- calibrated;
- run on accelerators.

This motivates JAX, XLA, PyTorch, and differentiable programming approaches.

A future notebook could compare a C++ Monte Carlo kernel with a JAX implementation for Greeks via automatic differentiation.

### 15.6 Event-driven market microstructure kernels

Market microstructure simulation is often difficult to vectorise because events arrive asynchronously.

C++ is especially useful for:

- limit order book reconstruction;
- order matching;
- queue position updates;
- latency modelling;
- stateful execution simulation.

A future Phase 6 notebook could implement a C++ limit-order-book replay engine and expose it to Python for analysis.

## 16. Suggested follow-up notebooks

This notebook naturally leads to:

1. **01_06_gpu_accelerated_backends_benchmark**  
   Comparing CPU NumPy, Numba, CuPy, and JAX for high-volume feature computation.

2. **02_13_multilevel_monte_carlo_pricing**  
   Moving Monte Carlo path simulation into faster kernels.

3. **05_01_vectorized_backtest_engine**  
   Building a full vectorised backtesting engine with transaction costs and slippage.

4. **05_02_event_driven_backtest_engine**  
   Handling stateful events that are harder to vectorise.

5. **06_06_microstructure_friction_cpp_core**  
   Using C++ to model limit-board locks, margin, and deadband rebalancing.

6. **06_11_cpp_python_bindings_pybind11**  
   Turning the demonstration kernel into a proper Python extension module.

7. **07_01_multi_asset_cta_research_pipeline**  
   Using compiled kernels inside a full research pipeline.

## 17. Summary

This notebook showed how to move from a Python finance prototype to a compiled C++ kernel.

The workflow was:

1. define a clear kernel specification;
2. implement a readable Python reference;
3. implement a vectorised NumPy version;
4. write the same logic in C++;
5. compile it into a shared library;
6. call it from Python;
7. validate numerical equivalence;
8. benchmark the alternatives.

The key computational takeaway is:

> C++ should not replace Python research workflows. It should accelerate carefully selected bottlenecks after correctness has been established.

The key financial takeaway is:

> Backtesting, pricing, risk, and execution systems often contain stateful numerical kernels. These kernels must be fast, but they must also be auditable and mathematically consistent.

## 18. Further reading

### Python/C++ integration

- pybind11 documentation.
- scikit-build-core documentation.
- Python `ctypes` documentation.
- CMake documentation.

### High-performance Python

- Numba documentation.
- NumPy documentation.
- JAX documentation.
- Cython documentation.

### Numerical and financial computing

- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*
- Joshi, M. *C++ Design Patterns and Derivatives Pricing.*
- Trefethen, L. N. and Bau, D. *Numerical Linear Algebra.*
- Boyd, S. and Vandenberghe, L. *Convex Optimization.*

### Software engineering for quant research

- Unit testing numerical code.
- Continuous integration for scientific Python packages.
- Reproducible builds and pinned environments.
- Benchmarking methodology for numerical software.